# 🎙️ Hindi Audiobook TTS Pipeline v7
### Latin → Devanagari → Edge-TTS Audio + SRT
**Pipeline:** Upload TXT file (Latin-script Hindi) → Qwen3.5:27b converts to Devanagari + assigns voices/prosody → edge-tts generates audio → SRT subtitle file

---
**⚠️ BEFORE RUNNING:**
- Runtime → Change runtime type → **T4 GPU**
- Qwen3.5:27b needs ~15GB VRAM — right at T4's limit. If it OOMs, the notebook auto-falls back to `qwen2.5:14b`
- Ollama model download takes **10–20 minutes** on first run

**Changes from v6:**
- 📁 TXT file upload (no more pasting text)
- 🎭 Research-backed prosody presets (natural, not robotic)
- 📖 Smart chunking for full book chapters (2000 chars)
- 🎯 Better few-shot prompting for emotional arc
- 🆕 New emotions: narrator_dramatic, pleading, commanding
---


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install all dependencies                          ║
# ╚══════════════════════════════════════════════════════════════╝
print('📦 Installing dependencies...')

!pip install -q edge-tts aiofiles requests
!sudo apt-get install -y -q ffmpeg pciutils

import subprocess, os, sys
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('✅ ffmpeg:', result.stdout.split('\n')[0])

import edge_tts
print('✅ edge-tts ready')
print('\n🎉 All dependencies installed!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install Ollama + start server                     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, time, os, requests

print('🦙 Installing Ollama and Python library...')

!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
!pip install -q ollama
!curl -fsSL https://ollama.com/install.sh | sh

print('\n🚀 Starting Ollama server with T4-optimized settings...')

# ── T4 Memory Optimizations ──────────────────────────────────────
# q4_0 KV cache uses ~50% less VRAM than q8_0, crucial for 27B on T4
os.environ['OLLAMA_HOST']            = '127.0.0.1:11434'
os.environ['OLLAMA_KEEP_ALIVE']      = '24h'
os.environ['CUDA_VISIBLE_DEVICES']   = '0'
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']   = 'q4_0'
os.environ['OLLAMA_NUM_PARALLEL']    = '1'       # prevent VRAM fragmentation

subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print('  Waiting for server...', end='')
server_ready = False
for _i in range(60):
    try:
        r = requests.get('http://127.0.0.1:11434/', timeout=2)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(1)

print()
if not server_ready:
    raise RuntimeError('❌ Ollama server did not start within 60s. Re-run this cell.')

print('✅ Ollama server running with Flash Attention + q4_0 KV cache!')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU — CPU mode'


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Pull model + warm it into VRAM                   ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, time
from ollama import chat as ollama_chat

PRIMARY_MODEL  = 'qwen3.5:27b'
FALLBACK_MODEL = 'qwen2.5:14b'
ACTIVE_MODEL   = None

# ── Shared inference options — tuned for T4 16GB ──────────────────
# num_ctx=4096: keeps KV cache small (~0.5GB in q4_0 vs ~4GB at 32K default)
# num_predict=2048: limits max output tokens (a 2000-char chunk rarely needs more)
# These two settings alone can take inference from 25min/chunk to 1-3min/chunk
# by keeping the entire model in VRAM instead of CPU-offloading.
OLLAMA_OPTIONS = {
    'num_ctx'          : 4096,
    'num_predict'      : 2048,
    'temperature'      : 0.7,
    'top_k'            : 20,
    'top_p'            : 0.9,
    'presence_penalty' : 1.0,
    'num_gpu'          : 99,     # force all layers to GPU
}

def pull_model(model_name):
    print(f'📥 Pulling {model_name}... (10-20 min first run)')
    start = time.time()
    result = subprocess.run(['ollama', 'pull', model_name],
                            capture_output=True, text=True)
    elapsed = int(time.time() - start)
    if result.returncode == 0:
        print(f'✅ {model_name} pulled in {elapsed}s')
        return True
    print(f'❌ Pull failed: {result.stderr[-300:]}')
    return False

def warmup_model(model_name):
    print(f'🔥 Warming up {model_name} into VRAM (may take 3-5 min first time)...')
    start = time.time()
    try:
        response_text = ''
        stream = ollama_chat(
            model=model_name,
            messages=[{'role': 'user', 'content': 'Reply with one word: READY'}],
            think=False,
            stream=True,
            options={**OLLAMA_OPTIONS, 'num_predict': 5},
        )
        for chunk in stream:
            response_text += chunk.message.content or ''
        elapsed = int(time.time() - start)
        print(f'✅ Model warm! Loaded into VRAM in {elapsed}s')
        print(f'   Warm-up response: "{response_text.strip()}"')
        return True
    except Exception as e:
        print(f'❌ Warm-up failed: {e}')
        return False

if pull_model(PRIMARY_MODEL):
    ACTIVE_MODEL = PRIMARY_MODEL
else:
    print(f'⚠️  Trying fallback: {FALLBACK_MODEL}')
    if pull_model(FALLBACK_MODEL):
        ACTIVE_MODEL = FALLBACK_MODEL
    else:
        raise RuntimeError('❌ Could not pull any model.')

print(f'\n🤖 Active model: {ACTIVE_MODEL}')
if not warmup_model(ACTIVE_MODEL):
    print('⚠️  Warm-up failed but continuing.')

# Show VRAM after warmup
print('\n📊 VRAM after model load:')
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU info'
!ollama ps 2>/dev/null || true

print(f'\n✅ Model ready. Proceed to Cell 4.')
print(f'   Inference settings: num_ctx={OLLAMA_OPTIONS["num_ctx"]}, '
      f'num_predict={OLLAMA_OPTIONS["num_predict"]}, '
      f'temperature={OLLAMA_OPTIONS["temperature"]}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Upload TXT file or paste text                    ║
# ╚══════════════════════════════════════════════════════════════╝

# ─── Configuration ────────────────────────────────────────────────
BOOK_NAME    = "My_Audiobook"
CHAPTER_NAME = "Chapter_01"

# ─── File Upload ──────────────────────────────────────────────────
INPUT_TEXT = ""
upload_success = False

try:
    from google.colab import files
    print("📁 Upload your TXT file (Latin-script Hindi/Hinglish):")
    print("   (Click 'Choose Files' below)")
    uploaded = files.upload()
    
    if uploaded:
        filename = list(uploaded.keys())[0]
        raw_bytes = uploaded[filename]
        # Try UTF-8 first, then latin-1 fallback
        try:
            INPUT_TEXT = raw_bytes.decode('utf-8')
        except UnicodeDecodeError:
            INPUT_TEXT = raw_bytes.decode('latin-1')
        upload_success = True
        print(f'\n✅ Loaded: {filename}')
    else:
        print('⚠️  No file uploaded.')
except ImportError:
    print('⚠️  Not running on Colab — paste text below instead.')
except Exception as e:
    print(f'⚠️  Upload failed: {e}')

# ─── Fallback: paste text ─────────────────────────────────────────
if not upload_success:
    print('\n📝 Paste your text between the triple quotes below and re-run:')
    INPUT_TEXT = """
Ek din aisa hua tha.
Buddha jannat mein kamal taal ke kinare akele chal rahe the.
Taal mein khilte hue kamal bilkul jade ki tarah safed the, aur unke sone ke kesar se ek dilkash khushboo hawa mein fail rahi thi.
Lagta hai jannat mein subah ho rahi thi.
Thodi der baad, Buddha taal ke kinare ruk gaye aur kamal ke patton ko chhan kar neeche dekhne lage.
Yeh kamal taal seedhe narak ki gehrayi par bana hua tha, aur saaf paani se unhone telescope jaise River of Three Crossings (teen crossing wali nadi) aur Mountain of Needles (kaanthon ka pahad) dekha.
Phir unhone Kandata naam ke ek aadmi ko narak mein doosron ke saath tadapte hue dekh liya.
Yeh Kandata, ek badnaam chor tha jisne har tarah ki buraiyan ki thi – logon ko maara aur gharon ko aag laga di.
Lekin uske khate mein ek achha kaam bhi tha.
Ek baar woh jungle se guzarte waqt usko ek chhota makdi raste par chalte hue dikha.
Kandata use machalane wala hi tha, lekin phir socha,
“Nahi yaar, yeh bhi toh jaan hai,”
aur usne usse jaane diya.
Buddha ko yeh yaad aa gaya aur unhone socha,
“Main iski madad kar sakta hoon.”
To unhone jannat ke kamal taal mein ek makdi dhunda aur uska dhaaga narak tak utaar diya.
Makdi ka dhaaga patla tha lekin mazboot, aur seedha narak mein chala gaya.
Kandata, jo narak mein dard se karah raha tha, usne achanak woh dhaaga dekha.
“Agar main is par charhunga toh pakka narak se bahar aa jaaunga,”
usne socha aur poori taqat se dhaage par chadhna shuru kar diya.
Lekin jab woh thoda upar chadha, to neeche dekh ke usko doosre paapi bhi wahi dhaaga charhte hue dikhe.
Yeh dekhte hi Kandata chillaya,
“Yeh makdi ka dhaaga mera hai!
Utar jao! Utar jao!”
Us pal hi, *plink* ki awaaz ke saath makdi ka dhaaga toot gaya.
Kandata aur doosre paapi sab narak mein gir gaye.
Jannat ke kamal taal ke kinare khade Buddha ne yeh dekha, unke chehre par udaasi thi, aur phir woh chupchap chalne lage.
"""

# ─── Cleanup & display ───────────────────────────────────────────
INPUT_TEXT = INPUT_TEXT.strip()
if not INPUT_TEXT:
    raise ValueError("❌ No text provided! Upload a TXT file or paste text above.")

print(f'\n📖 Input text loaded: {len(INPUT_TEXT)} characters')
print(f'\nPreview (first 300 chars):')
print(INPUT_TEXT[:300] + ('...' if len(INPUT_TEXT) > 300 else ''))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Voices, prosody presets, master prompt (v7)      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Voice assignments ─────────────────────────────────────────────
VOICES = {
    'narrator'      : 'hi-IN-SwaraNeural',
    'char_male'     : 'hi-IN-MadhurNeural',
    'char_female'   : 'hi-IN-SwaraNeural',
    'char_child'    : 'hi-IN-SwaraNeural',
    'char_elder'    : 'hi-IN-MadhurNeural',
    'inner_thought' : 'hi-IN-SwaraNeural',
}

# ── Prosody presets (v7 — research-calibrated) ────────────────────
# Research: Edge TTS sounds best within +-15% rate, +-30Hz pitch, +-15% vol.
# Values beyond these ranges sound increasingly artificial.
# Format: (rate, pitch, volume)
PRESETS = {
    # ── NARRATOR emotions ──────────────────────────────────────────
    'narrator_calm'     : ('-3%',   '+0Hz',   '+0%'),    # slight gravitas
    'narrator_slow'     : ('-12%',  '-3Hz',   '-2%'),    # dramatic build-up
    'narrator_dramatic' : ('-8%',   '+5Hz',   '+5%'),    # scene transitions, cliffhangers
    'tension'           : ('+5%',   '+8Hz',   '+5%'),    # suspense, danger approaching
    'relief'            : ('-6%',   '-3Hz',   '+0%'),    # peace restored

    # ── CHARACTER emotions ─────────────────────────────────────────
    'dialogue_normal' : ('+3%',   '+0Hz',   '+0%'),    # calm conversation
    'excited'         : ('+15%',  '+20Hz',  '+10%'),   # joy, surprise
    'sad'             : ('-15%',  '-15Hz',  '-8%'),    # grief, loss
    'angry'           : ('+12%',  '+15Hz',  '+12%'),   # rage, confrontation
    'whisper'         : ('-10%',  '-10Hz',  '-25%'),   # secret, danger
    'scared'          : ('+12%',  '+18Hz',  '-5%'),    # fear, panic
    'sarcastic'       : ('-8%',   '-8Hz',   '+3%'),    # irony, dry humor
    'child_happy'     : ('+10%',  '+40Hz',  '+5%'),    # playful child
    'elder_wise'      : ('-12%',  '-25Hz',  '-3%'),    # slow, deliberate
    'inner_thought'   : ('-8%',   '-8Hz',   '-15%'),   # reflective, quiet
    'humorous'        : ('+8%',   '+12Hz',  '+5%'),    # funny, lighthearted
    'pleading'        : ('-5%',   '+10Hz',  '-5%'),    # begging, emotional
    'commanding'      : ('+5%',   '-5Hz',   '+12%'),   # authority, orders
}

# ── Emotion sets for validation ───────────────────────────────────
NARRATOR_EMOTIONS  = {'narrator_calm', 'narrator_slow', 'narrator_dramatic',
                      'tension', 'relief'}
CHARACTER_EMOTIONS = {'dialogue_normal', 'excited', 'sad', 'angry', 'whisper',
                      'scared', 'sarcastic', 'child_happy', 'elder_wise',
                      'inner_thought', 'humorous', 'pleading', 'commanding'}

# ── MASTER SYSTEM PROMPT (v7 — multi-scene few-shot) ──────────────
with open('/tmp/system_prompt.txt', 'w', encoding='utf-8') as f:
    f.write('''You are an expert Hindi audiobook director and TTS script engineer. You receive Hindi story text (in Latin script) and convert it into a precise JSON script for edge-tts narration.

### STEP 1: LANGUAGE CONVERSION
Convert ALL Latin-script Hindi/Hinglish to Devanagari.
Examples: "ek" -> "एक", "bahut" -> "बहुत", "chal rahe the" -> "चल रहे थे"

KEEP these in original English (edge-tts pronounces them better):
- Brand names: Google, YouTube, WhatsApp, Amazon, Microsoft
- Tech terms: AI, TTS, WiFi, app, laptop, computer, internet
- Common English words used in Indian speech: "interesting", "actually", "curious"
- Proper nouns that are universally known in English

Numbers -> Devanagari words: "3" -> "तीन", "1947" -> "उन्नीस सौ सैंतालीस"

### STEP 2: CHARACTER ASSIGNMENT — THE MOST IMPORTANT RULE
Apply this 3-step test to EVERY sentence:

TEST: Is this text inside quotation marks ("...", '...', or "...") ?
 NO  -> character: "narrator"   (MANDATORY — no exceptions)
 YES -> Identify the speaker:
        - Male character speaking    -> "char_male"
        - Female character speaking  -> "char_female"
        - Child speaking             -> "char_child"
        - Elder (60+) speaking       -> "char_elder"
        - Someone THINKING (quoted)  -> "inner_thought"

CRITICAL RULES:
- A sentence describing a character is STILL narration -> "narrator"
- "Buddha walked", "Kandata saw" — ALL narrator, even if about a character
- ONLY actual quoted words get a character type
- When unsure, ALWAYS default to "narrator"

### STEP 3: EMOTION ASSIGNMENT — MAINTAIN EMOTIONAL ARC
IMPORTANT: Emotions should flow naturally through the story.
If tension is building, keep using "tension" until the climax.
Don't randomly switch back to "narrator_calm" mid-scene.
Think of the story as having an emotional trajectory.

Narrator segments -> ONLY narrator emotions:
  "narrator_calm"     -> default narration, scene description
  "narrator_slow"     -> dramatic reveal, important moment, scene change
  "narrator_dramatic" -> cliffhanger, major scene transition, twist
  "tension"           -> danger, suspense, something bad coming
  "relief"            -> resolution, safety, peace

Character segments -> character emotions:
  "dialogue_normal" -> calm speech
  "excited"         -> joy, surprise, triumph
  "sad"             -> grief, regret, loss
  "angry"           -> rage, frustration
  "whisper"         -> secret, danger nearby
  "scared"          -> fear, horror, panic
  "sarcastic"       -> irony, teasing
  "child_happy"     -> playful child energy
  "elder_wise"      -> slow, deliberate wisdom
  "inner_thought"   -> reflective, personal
  "humorous"        -> funny, lighthearted
  "pleading"        -> begging, emotional request
  "commanding"      -> authority, giving orders

NEVER assign character emotions to narrator segments or vice versa.

### STEP 4: PAUSE ASSIGNMENT (pause_after in milliseconds)
1000ms -> chapter break, major scene change, dramatic silence
800ms  -> paragraph break, scene transition
600ms  -> after important reveal, after "!" or "?"
500ms  -> end of emotional speech
400ms  -> normal end of sentence
300ms  -> between short action lines
200ms  -> after dialogue tag ("usne kaha,") before quoted speech
700ms  -> after dramatic "..." ellipsis in text
150ms  -> between rapid-fire dialogue exchanges

### STEP 5: SEGMENTATION RULES
- Max 2 sentences per segment (shorter = better prosody)
- Split narrator introduction from the quote it introduces:
  "Usne kaha," -> narrator segment (pause_after: 200)
  "Yeh mera hai!" -> char_male segment (pause_after: 500)
- Never merge narrator text with dialogue in the same segment
- Never cut a sentence in half

### FEW-SHOT EXAMPLES — Study these carefully

EXAMPLE 1 — ACTION/TENSION SCENE:
Input: "Raghu jungle mein chal raha tha. Achanak usne ek sher dekha. Woh dar gaya aur socha, 'Mujhe bhagna chahiye.' Phir woh tez bhaaga."

Output:
[
  {"id": 1, "character": "narrator", "emotion": "narrator_calm", "text": "रघु जंगल में चल रहा था।", "pause_after": 400},
  {"id": 2, "character": "narrator", "emotion": "tension", "text": "अचानक उसने एक शेर देखा।", "pause_after": 600},
  {"id": 3, "character": "narrator", "emotion": "tension", "text": "वह डर गया और सोचा,", "pause_after": 200},
  {"id": 4, "character": "inner_thought", "emotion": "scared", "text": "मुझे भागना चाहिए।", "pause_after": 500},
  {"id": 5, "character": "narrator", "emotion": "tension", "text": "फिर वह तेज़ भागा।", "pause_after": 800}
]
WHY: Tension builds from segment 2 and STAYS through segment 5. It doesn't drop back to calm.

EXAMPLE 2 — EMOTIONAL/SAD SCENE:
Input: "Seeta ki aankhon mein aansoo the. Usne dhire se kaha, 'Main tumhe kabhi nahi bhooluungi.' Phir woh chup ho gayi."

Output:
[
  {"id": 1, "character": "narrator", "emotion": "narrator_slow", "text": "सीता की आँखों में आँसू थे।", "pause_after": 600},
  {"id": 2, "character": "narrator", "emotion": "narrator_slow", "text": "उसने धीरे से कहा,", "pause_after": 200},
  {"id": 3, "character": "char_female", "emotion": "sad", "text": "मैं तुम्हें कभी नहीं भूलूँगी।", "pause_after": 700},
  {"id": 4, "character": "narrator", "emotion": "relief", "text": "फिर वह चुप हो गई।", "pause_after": 800}
]
WHY: narrator_slow sets the sad mood. The dialogue uses "sad". Relief comes at the quiet ending.

EXAMPLE 3 — MULTI-CHARACTER DIALOGUE:
Input: "Raju ne kaha, 'Chal yaar, chalte hain!' Guddi boli, 'Nahi, mujhe dar lag raha hai.' Dadaji ne muskuraate hue kaha, 'Daro mat bachcho, main hoon na.'"

Output:
[
  {"id": 1, "character": "narrator", "emotion": "narrator_calm", "text": "राजू ने कहा,", "pause_after": 200},
  {"id": 2, "character": "char_male", "emotion": "excited", "text": "चल यार, चलते हैं!", "pause_after": 400},
  {"id": 3, "character": "narrator", "emotion": "narrator_calm", "text": "गुड्डी बोली,", "pause_after": 200},
  {"id": 4, "character": "char_female", "emotion": "scared", "text": "नहीं, मुझे डर लग रहा है।", "pause_after": 500},
  {"id": 5, "character": "narrator", "emotion": "narrator_calm", "text": "दादाजी ने मुस्कुराते हुए कहा,", "pause_after": 200},
  {"id": 6, "character": "char_elder", "emotion": "elder_wise", "text": "डरो मत बच्चों, मैं हूँ ना।", "pause_after": 600}
]
WHY: Each character gets their own emotion. Elder uses elder_wise. Female child uses scared.

### OUTPUT FORMAT
Output ONLY a raw JSON array. No markdown, no code fences, no explanation.
Start directly with [ and end with ]

Each object:
{
  "id": <integer>,
  "character": <one of: "narrator","char_male","char_female","char_child","char_elder","inner_thought">,
  "emotion": <see allowed emotions above>,
  "text": <Devanagari text only>,
  "pause_after": <integer milliseconds>
}''')

with open('/tmp/system_prompt.txt', 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

print("✅ VOICES, PRESETS, SYSTEM_PROMPT defined (v7)")
print(f"   Narrator emotions : {len(NARRATOR_EMOTIONS)}")
print(f"   Character emotions: {len(CHARACTER_EMOTIONS)}")
print(f"   Voice roles       : {len(VOICES)}")
print(f"   Prosody presets   : {len(PRESETS)}")
print(f"   System prompt     : {len(SYSTEM_PROMPT)} chars")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Send text to Qwen3.5, get structured script      ║
# ╚══════════════════════════════════════════════════════════════╝
import json, re, time, textwrap
from ollama import chat as ollama_chat

# ── Pydantic schema for guaranteed valid JSON output ──────────────
try:
    from pydantic import BaseModel, RootModel
    from typing import Literal, List
    PYDANTIC_OK = True

    class Segment(BaseModel):
        id: int
        character: Literal[
            'narrator', 'char_male', 'char_female',
            'char_child', 'char_elder', 'inner_thought'
        ]
        emotion: Literal[
            'narrator_calm', 'narrator_slow', 'narrator_dramatic',
            'tension', 'relief',
            'dialogue_normal', 'excited', 'sad', 'angry', 'whisper',
            'scared', 'sarcastic', 'child_happy', 'elder_wise',
            'inner_thought', 'humorous', 'pleading', 'commanding'
        ]
        text: str
        pause_after: int

    class Script(RootModel):
        root: List[Segment]

    SCHEMA = Script.model_json_schema()
    print("✅ Pydantic schema loaded — Ollama will enforce valid enum values")

except ImportError:
    PYDANTIC_OK = False
    SCHEMA = None
    print("⚠️  Pydantic not found — install with: pip install pydantic")


def call_qwen(text_chunk, system_prompt, model, chunk_index=0, total_chunks=1,
              prev_context=''):
    '''Call Qwen3.5 with schema enforcement, emotional continuity, and speed-optimized options.'''
    context_hint = ""
    if total_chunks > 1:
        context_hint = (
            f"[Chunk {chunk_index+1} of {total_chunks}. "
            "Maintain consistent narrator voice for all non-quoted text. "
            "Maintain emotional arc — if the previous chunk ended with tension, "
            "continue that mood unless the text shifts. "
            "Only quoted speech/thoughts get character types.]\n\n"
        )
    if prev_context:
        context_hint += (
            f"[Previous chunk ended with: \"{prev_context}\" — "
            "maintain emotional continuity.]\n\n"
        )

    full_content = []
    stream = ollama_chat(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {
                'role': 'user',
                'content': (
                    context_hint +
                    'Convert the following text to the JSON script format. '
                    'Remember: ALL non-quoted text = narrator. '
                    'Maintain emotional arc across the story. '
                    'CRITICAL: Escape all double quotes inside text values with backslash. '
                    'Output ONLY the raw JSON array starting with [\n\n'
                    + text_chunk
                )
            },
        ],
        think=False,
        stream=True,
        format=SCHEMA,
        options=OLLAMA_OPTIONS,   # ← uses the shared speed-optimized options from Cell 3
    )
    for chunk in stream:
        full_content.append(chunk.message.content or '')
    return ''.join(full_content)


def repair_json(raw_json):
    '''
    Attempt to repair malformed JSON from LLM output.

    Common issues:
    1. Unescaped double quotes inside string values (Hindi text with quotes)
    2. Truncated output (missing closing brackets)
    3. Invalid enum values that schema enforcement missed
    '''
    # Strategy 1: Try parsing as-is first
    try:
        return json.loads(raw_json)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Extract individual JSON objects with regex and rebuild
    segments = []
    obj_pattern = re.compile(
        r'\{\s*'
        r'"id"\s*:\s*(\d+)\s*,\s*'
        r'"character"\s*:\s*"([^"]*?)"\s*,\s*'
        r'"emotion"\s*:\s*"([^"]*?)"\s*,\s*'
        r'"text"\s*:\s*"(.*?)"\s*,\s*'
        r'"pause_after"\s*:\s*(\d+)\s*'
        r'\}',
        re.DOTALL
    )

    matches = obj_pattern.findall(raw_json)
    if matches:
        for m in matches:
            seg_id, character, emotion, text, pause = m
            text = text.replace('\\"', '"').replace('"', '')
            segments.append({
                'id': int(seg_id),
                'character': character,
                'emotion': emotion,
                'text': text.strip(),
                'pause_after': int(pause)
            })
        if segments:
            print(f'  🔧 Regex extraction recovered {len(segments)} segments')
            return segments

    # Strategy 3: Fix unescaped quotes by processing character by character
    fixed = _fix_text_quotes(raw_json)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # Strategy 4: Extract whatever valid objects exist before the error point
    for i in range(len(raw_json) - 1, 0, -1):
        if raw_json[i] == '}':
            candidate = raw_json[:i+1] + ']'
            try:
                result = json.loads(candidate)
                if result:
                    print(f'  🔧 Truncation recovery: got {len(result)} segments')
                    return result
            except json.JSONDecodeError:
                continue

    raise ValueError(f'Could not repair JSON. First 400 chars:\n{raw_json[:400]}')


def _fix_text_quotes(raw_json):
    '''Fix unescaped double quotes inside "text" field values.'''
    result = []
    i = 0
    text_key = '"text"'

    while i < len(raw_json):
        pos = raw_json.find(text_key, i)
        if pos == -1:
            result.append(raw_json[i:])
            break

        result.append(raw_json[i:pos + len(text_key)])
        i = pos + len(text_key)

        while i < len(raw_json) and raw_json[i] in ' \t\n\r:':
            result.append(raw_json[i])
            i += 1

        if i >= len(raw_json) or raw_json[i] != '"':
            continue

        result.append('"')
        i += 1

        text_content = []
        while i < len(raw_json):
            ch = raw_json[i]

            if ch == '\\' and i + 1 < len(raw_json):
                text_content.append(ch)
                text_content.append(raw_json[i + 1])
                i += 2
                continue

            if ch == '"':
                rest = raw_json[i+1:].lstrip()
                if rest and rest[0] in ',}':
                    result.append(''.join(text_content))
                    result.append('"')
                    i += 1
                    break
                else:
                    text_content.append('\\"')
                    i += 1
                    continue

            text_content.append(ch)
            i += 1

    return ''.join(result)


def extract_json(raw_text):
    '''Extract and validate JSON array from model output — with robust repair.'''
    raw_text = re.sub(r'<think>[\s\S]*?</think>', '', raw_text, flags=re.IGNORECASE)
    raw_text = re.sub(r'```json\s*', '', raw_text)
    raw_text = re.sub(r'```\s*', '', raw_text)
    raw_text = raw_text.strip()

    start = raw_text.find('[')
    end   = raw_text.rfind(']') + 1
    if start == -1 or end == 0:
        raise ValueError(f'No JSON array found.\nRaw: {raw_text[:400]}')

    json_str = raw_text[start:end]
    return repair_json(json_str)


def auto_correct_emotions(segments):
    '''Post-processing: enforce emotion-to-character consistency + fix invalid emotions.'''
    VALID_ALL = NARRATOR_EMOTIONS | CHARACTER_EMOTIONS
    EMOTION_FIXES = {
        'dramatic'       : 'narrator_dramatic',
        'calm'           : 'narrator_calm',
        'slow'           : 'narrator_slow',
        'normal'         : 'dialogue_normal',
        'happy'          : 'excited',
        'fear'           : 'scared',
        'surprise'       : 'excited',
        'neutral'        : 'narrator_calm',
        'narration'      : 'narrator_calm',
        'narrative'      : 'narrator_calm',
        'narrator'       : 'narrator_calm',
        'thought'        : 'inner_thought',
        'thinking'       : 'inner_thought',
        'wise'           : 'elder_wise',
        'command'        : 'commanding',
        'plead'          : 'pleading',
        'humor'          : 'humorous',
        'funny'          : 'humorous',
        'joke'           : 'humorous',
        'cry'            : 'sad',
        'grief'          : 'sad',
        'rage'           : 'angry',
        'fury'           : 'angry',
        'shout'          : 'angry',
        'panic'          : 'scared',
        'suspense'       : 'tension',
        'tense'          : 'tension',
        'mysterious'     : 'tension',
        'soft'           : 'whisper',
        'gentle'         : 'relief',
        'peace'          : 'relief',
    }

    corrections = 0
    for seg in segments:
        char = seg.get('character', 'narrator')
        emo  = seg.get('emotion', 'narrator_calm')

        if emo not in VALID_ALL:
            fixed = EMOTION_FIXES.get(emo.lower(), None)
            if fixed:
                seg['emotion'] = fixed
                emo = fixed
                corrections += 1
            else:
                seg['emotion'] = 'narrator_calm' if char == 'narrator' else 'dialogue_normal'
                emo = seg['emotion']
                corrections += 1

        if char == 'narrator' and emo not in NARRATOR_EMOTIONS:
            seg['emotion'] = 'narrator_calm'
            corrections += 1
        elif char != 'narrator' and emo not in CHARACTER_EMOTIONS:
            seg['emotion'] = 'dialogue_normal'
            corrections += 1

    if corrections:
        print(f'  🔧 Auto-corrected {corrections} emotion issue(s)')
    return segments


def chunk_text(text, max_chars=1200):
    '''Smart chunking: split at paragraph/sentence boundaries.
    1200 chars keeps prompt + output within num_ctx=4096 comfortably.
    Smaller chunks = faster inference per chunk + better prosody granularity.
    '''
    paragraphs = [p.strip() for p in text.strip().split('\n') if p.strip()]
    chunks, current = [], ''

    for para in paragraphs:
        if len(para) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ''
            sentences = re.split(r'(?<=[.!?।])\s+', para)
            sent_buf = ''
            for sent in sentences:
                if len(sent_buf) + len(sent) > max_chars and sent_buf:
                    chunks.append(sent_buf.strip())
                    sent_buf = sent
                else:
                    sent_buf += ' ' + sent if sent_buf else sent
            if sent_buf.strip():
                chunks.append(sent_buf.strip())
        elif len(current) + len(para) + 1 > max_chars and current:
            chunks.append(current.strip())
            current = para
        else:
            current += '\n' + para if current else para

    if current.strip():
        chunks.append(current.strip())
    return chunks


# ── Main conversion loop (with retry + progress timing) ──────────
MAX_RETRIES_PER_CHUNK = 2

text_chunks = chunk_text(INPUT_TEXT, max_chars=1200)
total       = len(text_chunks)
print(f'📑 Text split into {total} chunk(s)')
print(f'   Schema enforcement: {"ON ✅" if PYDANTIC_OK else "OFF ⚠️"}')
print(f'   Inference: num_ctx={OLLAMA_OPTIONS["num_ctx"]}, num_predict={OLLAMA_OPTIONS["num_predict"]}')

all_segments, segment_id = [], 1
prev_context = ''
chunk_times = []

for i, chunk in enumerate(text_chunks):
    print(f'\n🤖 Chunk {i+1}/{total} ({len(chunk)} chars)...')
    start_t = time.time()
    raw = ''
    last_error = None

    for attempt in range(1, MAX_RETRIES_PER_CHUNK + 1):
        try:
            raw = call_qwen(chunk, SYSTEM_PROMPT, ACTIVE_MODEL,
                            chunk_index=i, total_chunks=total,
                            prev_context=prev_context)
            if not raw.strip():
                raise ValueError('Empty response — re-run Cell 3 to reload model')
            segments = extract_json(raw)
            segments = auto_correct_emotions(segments)
            segments = [s for s in segments if s.get('text', '').strip()]

            for seg in segments:
                seg['id'] = segment_id
                segment_id += 1
            all_segments.extend(segments)

            elapsed = time.time() - start_t
            chunk_times.append(elapsed)
            avg_time = sum(chunk_times) / len(chunk_times)
            remaining = avg_time * (total - i - 1)
            remaining_min = remaining / 60

            print(f'  ✅ {len(segments)} segments in {elapsed:.1f}s'
                  + (f' (attempt {attempt})' if attempt > 1 else ''))
            print(f'     ⏱️  ETA: {remaining_min:.1f} min remaining ({total-i-1} chunks)')

            if segments:
                last_texts = [s.get('text','') for s in segments[-2:]]
                prev_context = ' '.join(last_texts)[-200:]
            last_error = None
            break

        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES_PER_CHUNK:
                print(f'  ⚠️  Attempt {attempt} failed ({e}), retrying...')
                time.sleep(2)
            else:
                print(f'  ❌ Error on chunk {i+1} after {attempt} attempts: {e}')
                if raw:
                    print(f'  Raw output (first 500 chars):\n  {raw[:500]}')

    if last_error:
        raise last_error

total_time = sum(chunk_times)
print(f'\n✅ Total segments: {len(all_segments)}')
print(f'   Total processing time: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'   Average per chunk: {total_time/total:.1f}s')

# ── Character breakdown ───────────────────────────────────────────
char_counts = {}
for s in all_segments:
    char_counts[s.get('character','?')] = char_counts.get(s.get('character','?'), 0) + 1

print('\n📊 Character breakdown:')
for char, count in sorted(char_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(all_segments)
    print(f'   {char:<16}: {count:3d} segments ({pct:.0f}%)')

print('\n📋 First 8 segments:')
for seg in all_segments[:8]:
    print(f'  [{seg["id"]:02d}] {seg["character"]:<14} | {seg["emotion"]:<18} | {seg["text"][:55]}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Validate & map segments to TTS parameters        ║
# ╚══════════════════════════════════════════════════════════════╝

def resolve_voice(character):
    mapping = {
        'narrator'      : VOICES['narrator'],
        'char_male'     : VOICES['char_male'],
        'char_female'   : VOICES['char_female'],
        'char_child'    : VOICES['char_child'],
        'char_elder'    : VOICES['char_elder'],
        'inner_thought' : VOICES['inner_thought'],
    }
    return mapping.get(character, VOICES['narrator'])

def resolve_prosody(emotion, character):
    '''Return (rate, pitch, volume) — v7 research-calibrated.'''
    base = PRESETS.get(emotion, PRESETS['narrator_calm'])
    rate, pitch, volume = base

    # Character-specific pitch adjustments (v7: reduced from v6)
    if character == 'char_child':
        pitch_val = int(pitch.replace('Hz', '').replace('+', ''))
        pitch_val = min(pitch_val + 30, 100)   # v7: +30 (was +60)
        pitch = f'+{pitch_val}Hz' if pitch_val >= 0 else f'{pitch_val}Hz'

    if character == 'char_elder':
        pitch_val = int(pitch.replace('Hz', '').replace('+', ''))
        pitch_val = max(pitch_val - 15, -80)    # v7: -15 (was -30)
        pitch = f'{pitch_val}Hz' if pitch_val < 0 else f'+{pitch_val}Hz'

    return rate, pitch, volume

# Validate and enrich segments
VALID_EMOTIONS    = set(PRESETS.keys())
VALID_CHARACTERS  = {'narrator', 'char_male', 'char_female',
                     'char_child', 'char_elder', 'inner_thought'}

enriched = []
issues   = 0

for seg in all_segments:
    if seg.get('character') not in VALID_CHARACTERS:
        seg['character'] = 'narrator'
        issues += 1
    if seg.get('emotion') not in VALID_EMOTIONS:
        seg['emotion'] = 'narrator_calm'
        issues += 1
    if not seg.get('text', '').strip():
        continue

    voice              = resolve_voice(seg['character'])
    rate, pitch, vol   = resolve_prosody(seg['emotion'], seg['character'])
    pause              = int(seg.get('pause_after', 400))

    enriched.append({
        'id'          : seg['id'],
        'text'        : seg['text'].strip(),
        'character'   : seg['character'],
        'emotion'     : seg['emotion'],
        'voice'       : voice,
        'rate'        : rate,
        'pitch'       : pitch,
        'volume'      : vol,
        'pause_after' : pause,
    })

print(f'✅ Validated {len(enriched)} segments ({issues} field(s) auto-corrected)')
print('\n📋 Enriched segment sample:')
for s in enriched[:5]:
    print(f'  [{s["id"]:02d}] {s["character"]:<14} | {s["emotion"]:<18} | voice={s["voice"]}')
    print(f'       rate={s["rate"]:<7} pitch={s["pitch"]:<8} vol={s["volume"]}')
    print(f'       text: "{s["text"][:70]}"')
    print()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Generate audio for all segments (parallel)       ║
# ╚══════════════════════════════════════════════════════════════╝
import asyncio, edge_tts, os, time

AUDIO_DIR   = '/content/audio_chunks'
os.makedirs(AUDIO_DIR, exist_ok=True)

MAX_CONCURRENT = 4
MAX_RETRIES    = 3

async def generate_segment_audio(seg, semaphore):
    out_path = os.path.join(AUDIO_DIR, f'seg_{seg["id"]:04d}.mp3')
    srt_path = os.path.join(AUDIO_DIR, f'seg_{seg["id"]:04d}.srt')

    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                comm = edge_tts.Communicate(
                    seg['text'],
                    seg['voice'],
                    rate   = seg['rate'],
                    pitch  = seg['pitch'],
                    volume = seg['volume'],
                )
                submaker = edge_tts.SubMaker()
                with open(out_path, 'wb') as f:
                    async for chunk in comm.stream():
                        if chunk['type'] == 'audio':
                            f.write(chunk['data'])
                        elif chunk['type'] in ('WordBoundary', 'SentenceBoundary'):
                            submaker.feed(chunk)

                with open(srt_path, 'w', encoding='utf-8') as f:
                    f.write(submaker.get_srt())

                dur_proc = await asyncio.create_subprocess_exec(
                    'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                    '-of', 'default=noprint_wrappers=1:nokey=1', out_path,
                    stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.DEVNULL
                )
                dur_out, _ = await dur_proc.communicate()
                duration_ms = int(float(dur_out.decode().strip()) * 1000)

                return seg['id'], out_path, srt_path, duration_ms

            except Exception as e:
                if attempt == MAX_RETRIES:
                    print(f'  ❌ Seg {seg["id"]} FAILED after {MAX_RETRIES} attempts: {e}')
                    raise
                await asyncio.sleep(2 * attempt)

async def generate_all(segments):
    sem    = asyncio.Semaphore(MAX_CONCURRENT)
    tasks  = [generate_segment_audio(s, sem) for s in segments]
    total  = len(tasks)
    done   = 0
    results = []

    for coro in asyncio.as_completed(tasks):
        result = await coro
        results.append(result)
        done += 1
        seg_id = result[0]
        dur_s  = result[3] / 1000
        print(f'  ✓ [{done:03d}/{total}] seg_{seg_id:04d}  {dur_s:.1f}s', end='\r')

    print()
    results.sort(key=lambda x: x[0])
    return results

print(f'🎙️  Generating audio for {len(enriched)} segments...\n')
start_time = time.time()

audio_results = await generate_all(enriched)

total_audio_ms = sum(r[3] for r in audio_results)
total_time     = time.time() - start_time
print(f'\n✅ All audio generated!')
print(f'   Total audio duration : {total_audio_ms/1000:.1f}s ({total_audio_ms/60000:.1f} min)')
print(f'   Generation time      : {total_time:.1f}s')
print(f'   Speed ratio          : {total_audio_ms/1000/total_time:.1f}x realtime')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Merge audio + build SRT (with silence gaps)      ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, os, re
from pathlib import Path

OUTPUT_DIR  = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINAL_MP3 = os.path.join(OUTPUT_DIR, f'{BOOK_NAME}_{CHAPTER_NAME}.mp3')
FINAL_SRT = os.path.join(OUTPUT_DIR, f'{BOOK_NAME}_{CHAPTER_NAME}.srt')

# ── Generate silence files for pauses ─────────────────────────────
SILENCE_DIR = os.path.join(AUDIO_DIR, 'silence')
os.makedirs(SILENCE_DIR, exist_ok=True)

silence_cache = {}  # cache silence files by duration

def get_silence_file(pause_ms):
    '''Create a silence MP3 file of given duration, cached.'''
    # Round to nearest 50ms to reduce unique files
    rounded = max(50, (pause_ms // 50) * 50)
    if rounded in silence_cache:
        return silence_cache[rounded]

    silence_path = os.path.join(SILENCE_DIR, f'silence_{rounded}ms.mp3')
    if not os.path.exists(silence_path):
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i',
            f'anullsrc=r=24000:cl=mono',
            '-t', f'{rounded/1000:.3f}',
            '-c:a', 'libmp3lame', '-b:a', '48k',
            silence_path
        ], capture_output=True)
    silence_cache[rounded] = silence_path
    return silence_path

# ── Build concat list with silence gaps ───────────────────────────
concat_list = os.path.join(AUDIO_DIR, 'concat_list.txt')
with open(concat_list, 'w') as f:
    for idx, (seg_id, mp3_path, srt_path, dur_ms) in enumerate(audio_results):
        f.write(f"file '{mp3_path}'\n")
        # Add silence gap after each segment (except last)
        if idx < len(audio_results) - 1:
            seg_info = next((s for s in enriched if s['id'] == seg_id), {})
            pause_ms = seg_info.get('pause_after', 400)
            if pause_ms > 0:
                silence_file = get_silence_file(pause_ms)
                f.write(f"file '{silence_file}'\n")

print('🔗 Merging audio chunks with silence gaps...')
merge_result = subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', concat_list,
    '-acodec', 'libmp3lame', '-b:a', '128k',
    FINAL_MP3
], capture_output=True, text=True)

if merge_result.returncode != 0:
    print('❌ ffmpeg merge error:')
    print(merge_result.stderr[-500:])
    raise RuntimeError('Audio merge failed')

mp3_size = os.path.getsize(FINAL_MP3) // 1024
print(f'✅ Audio merged → {FINAL_MP3} ({mp3_size} KB)')

# ── Build master SRT with accurate timestamps ─────────────────────
def ms_to_srt_time(ms):
    ms   = max(0, int(ms))
    h    = ms // 3600000; ms %= 3600000
    m    = ms // 60000;   ms %= 60000
    s    = ms // 1000;    ms %= 1000
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def parse_srt(srt_text):
    entries = []
    blocks  = re.split(r'\n\n+', srt_text.strip())
    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) < 3:
            continue
        time_match = re.search(
            r'(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})',
            lines[1]
        )
        if not time_match:
            continue
        def parse_ts(ts):
            ts = ts.replace(',', '.')
            h, m, rest = ts.split(':')
            s, ms = rest.split('.')
            return int(h)*3600000 + int(m)*60000 + int(s)*1000 + int(ms[:3])
        start_ms = parse_ts(time_match.group(1))
        end_ms   = parse_ts(time_match.group(2))
        text     = ' '.join(lines[2:])
        entries.append((start_ms, end_ms, text))
    return entries

master_entries = []
time_offset_ms = 0
srt_index      = 1

for seg_id, mp3_path, srt_path, dur_ms in audio_results:
    seg_info = next((s for s in enriched if s['id'] == seg_id), {})

    try:
        with open(srt_path, 'r', encoding='utf-8') as f:
            srt_content = f.read().strip()
    except:
        srt_content = ''

    if srt_content:
        entries = parse_srt(srt_content)
        for start_ms, end_ms, text in entries:
            global_start = time_offset_ms + start_ms
            global_end   = time_offset_ms + end_ms
            master_entries.append((srt_index, global_start, global_end, text))
            srt_index += 1
    else:
        text = seg_info.get('text', '')
        if text:
            master_entries.append((srt_index, time_offset_ms,
                                   time_offset_ms + dur_ms, text))
            srt_index += 1

    pause_ms = seg_info.get('pause_after', 400)
    time_offset_ms += dur_ms + pause_ms

with open(FINAL_SRT, 'w', encoding='utf-8') as f:
    for idx, start_ms, end_ms, text in master_entries:
        f.write(f'{idx}\n')
        f.write(f'{ms_to_srt_time(start_ms)} --> {ms_to_srt_time(end_ms)}\n')
        f.write(f'{text}\n\n')

srt_size = os.path.getsize(FINAL_SRT) // 1024
total_duration_str = ms_to_srt_time(time_offset_ms)

print(f'✅ SRT file built → {FINAL_SRT}')
print(f'   Subtitle entries : {len(master_entries)}')
print(f'   Total duration   : {total_duration_str}')
print(f'   File size        : {srt_size} KB')

print('\n📋 SRT Preview (first 5 entries):')
for entry in master_entries[:5]:
    idx, s, e, t = entry
    print(f'  {idx}  {ms_to_srt_time(s)} --> {ms_to_srt_time(e)}')
    print(f'  {t}')
    print()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Preview audio + download files                  ║
# ╚══════════════════════════════════════════════════════════════╝
from IPython.display import Audio, display
from google.colab import files

# Audio preview
print('🎧 Audio preview (first 30 seconds):')
display(Audio(FINAL_MP3))

# Download files
print('\n📥 Download your files:')
try:
    files.download(FINAL_MP3)
    files.download(FINAL_SRT)
    print('✅ Downloads triggered!')
except Exception as e:
    print(f'⚠️  Auto-download failed: {e}')
    print(f'   MP3: {FINAL_MP3}')
    print(f'   SRT: {FINAL_SRT}')
    print('   Use the file browser (📁) on the left to download manually.')
